# Chapter 3 — Intensity Transformations and Spatial Filtering

این Notebook چند آزمایش مستقل برای تبدیل‌های شدت، هیستوگرام، هموارسازی، تیزسازی و فیلترهای فرکانسی-مکانی فصل سوم فراهم می‌کند.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, gaussian_filter, median_filter


In [ ]:
def synthetic_image(n=256):
    y,x=np.mgrid[0:n,0:n]
    f=0.12+0.55*x/n
    f += 0.25*(((x-70)**2+(y-175)**2)<38**2)
    f += 0.30*((x>145)&(x<220)&(y>35)&(y<115))
    f += 0.06*np.sin(x/6)*np.sin(y/10)
    return np.clip(f,0,1)

f=synthetic_image()
plt.imshow(f,cmap="gray"); plt.axis("off");


## 1) Point intensity transformations


In [ ]:
neg=1-f
log=np.log1p(9*f)/np.log(10)
gamma=f**0.45
thr=(f>=0.5).astype(float)
fig,axs=plt.subplots(1,5,figsize=(14,3))
for a,img,t in zip(axs,[f,neg,log,gamma,thr],["input","negative","log","gamma=.45","threshold"]):
    a.imshow(img,cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout();


## 2) Histogram equalization


In [ ]:
u=np.round(f*255).astype(np.uint8)
h=np.bincount(u.ravel(),minlength=256)
cdf=h.cumsum().astype(float); cdf=(cdf-cdf.min())/(cdf.max()-cdf.min()+1e-12)
eq=cdf[u]
fig,axs=plt.subplots(1,2,figsize=(7,3))
axs[0].imshow(f,cmap="gray"); axs[0].set_title("input")
axs[1].imshow(eq,cmap="gray"); axs[1].set_title("equalized")
for a in axs:a.axis("off")
plt.tight_layout();


## 3) Smoothing: box, Gaussian, median


In [ ]:
rng=np.random.default_rng(2)
noisy=f.copy(); r=rng.random(f.shape)
noisy[r<.025]=0; noisy[(r>=.025)&(r<.05)]=1
box=convolve(noisy,np.ones((5,5))/25,mode="reflect")
gauss=gaussian_filter(noisy,1.2)
med=median_filter(noisy,5,mode="reflect")
fig,axs=plt.subplots(1,4,figsize=(12,3))
for a,img,t in zip(axs,[noisy,box,gauss,med],["noisy","box","Gaussian","median"]):
    a.imshow(img,cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout();


## 4) Laplacian, high-boost and Sobel


In [ ]:
lapk=np.array([[0,1,0],[1,-4,1],[0,1,0]],float)
lap=convolve(f,lapk,mode="reflect")
lsharp=np.clip(f-lap,0,1)
blur=gaussian_filter(f,2); mask=f-blur; hb=np.clip(f+1.8*mask,0,1)
sx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]],float); sy=sx.T
sobel=np.hypot(convolve(f,sx,mode="reflect"),convolve(f,sy,mode="reflect"))
fig,axs=plt.subplots(1,5,figsize=(15,3))
for a,img,t in zip(axs,[f,lap,lsharp,hb,sobel],["input","Laplacian","sharpened","high-boost","Sobel"]):
    a.imshow(img,cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout();


## 5) Zone plate filter family


In [ ]:
n=320
y,x=np.mgrid[-1:1:complex(n),-1:1:complex(n)]
z=.5*(1+np.cos(120*np.pi*(x*x+y*y)))
lp1=gaussian_filter(z,2); lp2=gaussian_filter(z,7)
hp=z-lp1; br=lp2+(z-lp1); bp=z-br
fig,axs=plt.subplots(1,5,figsize=(15,3))
for a,img,t in zip(axs,[z,lp1,hp,br,bp],["zone plate","lowpass","highpass","bandreject","bandpass"]):
    a.imshow(img,cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout();
